# [LAB-04] 데이터 다루기

## 4. 결측치 처리

### #01 준비작업

1. 패키지 설치

In [60]:
!pip install --upgrade scikit-learn

2. 라이브러리 참조

In [61]:
import numpy as np
from jussam import load_data
from pandas import DataFrame
from sklearn.impute import SimpleImputer
from sklearn import set_config

set_config(display="text")

3. 데이터 가져오기

In [62]:
origin = load_data("ref_sample")
origin

📚 데이터 정제를 위한 실습용 데이터


,kor,eng,math,sic
name,,,,
철수,98.000,77,88.000,64.000
영희,88.000,120,62.000,72.000
민철,NaN,70,83.000,79.000
수현,63.000,60,31.000,71.000
호영,75.000,50,90.000,NaN
영호,80.000,88,91.000,72.000
용식,82.000,88,NaN,90.000
나영,90.000,92,81.000,NaN
석영,91.000,90,89.000,80.000


### #02 결측치 확인하기

1. 결측치 여부 확인

In [63]:
empty = origin.isnull()
empty

,kor,eng,math,sic
name,,,,
철수,False,False,False,False
영희,False,False,False,False
민철,True,False,False,False
수현,False,False,False,False
호영,False,False,False,True
영호,False,False,False,False
용식,False,False,True,False
나영,False,False,False,True
석영,False,False,False,False


2. 각 열별로 결측치의 수를 확인

In [64]:
empty.sum()

kor     1
eng     0
math    1
sic     2
dtype: int64

### #03 결측치 소거

1. 행 단위 삭제

In [65]:
na1 = origin.dropna()
na1

,kor,eng,math,sic
name,,,,
철수,98.000,77,88.000,64.000
영희,88.000,120,62.000,72.000
수현,63.000,60,31.000,71.000
영호,80.000,88,91.000,72.000
석영,91.000,90,89.000,80.000


2. 열 단위 삭제

In [66]:
na2 = origin.dropna(axis=1)
na2

,eng
name,
철수,77
영희,120
민철,70
수현,60
호영,50
영호,88
용식,88
나영,92
석영,90


3. 특정 열에 대해서만 결측치가 존재하는 행 삭제

In [67]:
na3 = origin.dropna(subset=['kor'])
na3

,kor,eng,math,sic
name,,,,
철수,98.000,77,88.000,64.000
영희,88.000,120,62.000,72.000
수현,63.000,60,31.000,71.000
호영,75.000,50,90.000,NaN
영호,80.000,88,91.000,72.000
용식,82.000,88,NaN,90.000
나영,90.000,92,81.000,NaN
석영,91.000,90,89.000,80.000


### #04 결측치 대체

1. 고정값으로 대체

In [68]:
re_df1 = origin.fillna(value=50)
re_df1

,kor,eng,math,sic
name,,,,
철수,98.000,77,88.000,64.000
영희,88.000,120,62.000,72.000
민철,50.000,70,83.000,79.000
수현,63.000,60,31.000,71.000
호영,75.000,50,90.000,50.000
영호,80.000,88,91.000,72.000
용식,82.000,88,50.000,90.000
나영,90.000,92,81.000,50.000
석영,91.000,90,89.000,80.000


2. 통계적 값으로 대체(1) - 결측치를 정제할 규칙을 담고 있는 객체 생성

In [69]:
imr = SimpleImputer(missing_values=np.nan, strategy='mean')
imr

SimpleImputer()

3. 통계적 값으로 대체(2) - 생성된 규칙을 적용

In [70]:
df_imr = imr.fit_transform(origin.values)
df_imr

array([[ 98.        ,  77.        ,  88.        ,  64.        ],
       [ 88.        , 120.        ,  62.        ,  72.        ],
       [ 83.375     ,  70.        ,  83.        ,  79.        ],
       [ 63.        ,  60.        ,  31.        ,  71.        ],
       [ 75.        ,  50.        ,  90.        ,  75.42857143],
       [ 80.        ,  88.        ,  91.        ,  72.        ],
       [ 82.        ,  88.        ,  76.875     ,  90.        ],
       [ 90.        ,  92.        ,  81.        ,  75.42857143],
       [ 91.        ,  90.        ,  89.        ,  80.        ]])

4. 통계적 값으로 대체(3) - 결측치 정제 결과를 DataFrame으로 재구성

In [71]:
re_df2 = DataFrame(df_imr,
                   index=origin.index,          # 원본의 인덱스
                   columns=origin.columns       # 원본의 컬럼명
                   )
re_df2

,kor,eng,math,sic
name,,,,
철수,98.000,77.000,88.000,64.000
영희,88.000,120.000,62.000,72.000
민철,83.375,70.000,83.000,79.000
수현,63.000,60.000,31.000,71.000
호영,75.000,50.000,90.000,75.429
영호,80.000,88.000,91.000,72.000
용식,82.000,88.000,76.875,90.000
나영,90.000,92.000,81.000,75.429
석영,91.000,90.000,89.000,80.000


5. 특정 컬럼의 값 하나만 처리하기(1) - 컬럼의 값만 추출 후, n행 1열로 재구성

In [72]:
reshape = origin['kor'].values.reshape(-1, 1)
reshape

array([[98.],
       [88.],
       [nan],
       [63.],
       [75.],
       [80.],
       [82.],
       [90.],
       [91.]])

6. 특정 컬럼의 값 하나만 처리하기(2) - 특정 컬럼의 값 하나만 처리하기

In [73]:
reshape_imr = imr.fit_transform(reshape)
reshape_imr

array([[98.   ],
       [88.   ],
       [83.375],
       [63.   ],
       [75.   ],
       [80.   ],
       [82.   ],
       [90.   ],
       [91.   ]])

7. 특정 컬럼의 값 하나만 처리하기(3) - 결측치 처리 결과를 DataFrame의 컬럼에 적용하기

In [74]:
특정컬럼_정제df = origin.copy()
특정컬럼_정제df['kor'] = reshape_imr
특정컬럼_정제df

,kor,eng,math,sic
name,,,,
철수,98.000,77,88.000,64.000
영희,88.000,120,62.000,72.000
민철,83.375,70,83.000,79.000
수현,63.000,60,31.000,71.000
호영,75.000,50,90.000,NaN
영호,80.000,88,91.000,72.000
용식,82.000,88,NaN,90.000
나영,90.000,92,81.000,NaN
석영,91.000,90,89.000,80.000


---

## 연습문제

### 상점 판매 데이터 정제 챌린지 🛒
안녕하세요, 데이터 분석가님!
최근 한 달간의 판매 기록(month_sales)을 받았는데, 아쉽게도 일부 데이터가 누락되었습니다.   
정확한 월말 매출 보고를 위해 분석가님의 도움이 필요합니다.   
아래의 미션을 순서대로 해결하여 데이터 정제를 완료하세요!   

1. 미션 1: 데이터 탐색
- 데이터를 불러온 후, 각 열(column)에 몇 개의 결측치(missing value)가 있는지 확인해주세요.   

2. 미션 2: 신뢰할 수 없는 데이터 제거
- Quantity(수량) 정보가 누락된 데이터는 주문 과정에서 오류가 있었을 가능성이 높습니다.
- 이 데이터는 분석에서 제외하기로 결정했습니다.
- Quantity 열에 결측치가 있는 모든 행(row)을 삭제해주세요.   

3. 미션 3: 가격 정보 복원
- 일부 제품의 Price(가격) 정보가 누락되었습니다.
- 해당 제품들은 Price 열의 전체 평균 가격으로 값을 채워 넣어 데이터를 복원해주세요.

In [ ]:
# 1. 라이브러리 참조
import numpy as np
from jussam import load_data
from sklearn.impute import SimpleImputer

# 2. 데이터 불러오기
origin = load_data('month_sales')
origin

📚 데이터 정제(결측치 처리) 연습문제용 데이터

    field     description
--  --------  ----------------
 0  OrderID   주문번호(인덱스)
 1  Product   상품명
 2  Category  상품 카테고리
 3  Price     단가
 4  Quantity  주문 수량



,Product,Category,Price,Quantity
OrderID,,,,
1001,Laptop,Electronics,1200.000,10.000
1002,Mouse,Electronics,25.000,NaN
1003,Keyboard,Electronics,NaN,5.000
1004,Monitor,Electronics,300.000,8.000
1005,Webcam,Electronics,80.000,NaN
1006,Desk,Furniture,150.000,3.000
1007,Chair,Furniture,NaN,2.000
1008,Lamp,Furniture,40.000,15.000
1009,Notebook,Stationery,5.000,100.000


In [76]:
# 3. 결측치 확인하기
empty = origin.isnull()
empty.sum()

Product     0
Category    0
Price       2
Quantity    3
dtype: int64

In [77]:
# 4. 신뢰할 수 없는 데이터 제거
cleaned_df = origin.dropna(subset=['Quantity'])
cleaned_df

,Product,Category,Price,Quantity
OrderID,,,,
1001,Laptop,Electronics,1200.000,10.000
1003,Keyboard,Electronics,NaN,5.000
1004,Monitor,Electronics,300.000,8.000
1006,Desk,Furniture,150.000,3.000
1007,Chair,Furniture,NaN,2.000
1008,Lamp,Furniture,40.000,15.000
1009,Notebook,Stationery,5.000,100.000


In [ ]:
# 5. 가격 정보 복원
reshape = cleaned_df['Price'].values.reshape(-1, 1)

imr = SimpleImputer(missing_values=np.nan, strategy='mean')
reshape_imr = imr.fit_transform(reshape)

cleaned_df['Price'] = reshape_imr
cleaned_df

,Product,Category,Price,Quantity
OrderID,,,,
1001,Laptop,Electronics,1200.000,10.000
1003,Keyboard,Electronics,339.000,5.000
1004,Monitor,Electronics,300.000,8.000
1006,Desk,Furniture,150.000,3.000
1007,Chair,Furniture,339.000,2.000
1008,Lamp,Furniture,40.000,15.000
1009,Notebook,Stationery,5.000,100.000
